In [5]:
# ============================================================
# CELL 1: CREATE AND OPEN THE SG-5 WORKING FOLDER
# ============================================================
# This creates a folder named "module_sg5" inside Colab's
# working directory and then moves into that folder.
#
# All SG-5 Python files will be created and run from here.
# ============================================================

!mkdir -p /content/module_sg5
%cd /content/module_sg5

/content/module_sg5


In [6]:
# ============================================================
# CELL 2: CREATE MOCK SG-4 INPUTS FOR SG-5 TESTING
# ============================================================
# These dictionaries simulate the temporal output that SG-5
# would normally receive from SG-4.
#
# We use them so SG-5 can be tested independently before
# full integration with SG-4 and the Jetson system.
# ============================================================

%%writefile mock_inputs.py

# ------------------------------------------------------------
# CASE 1: NORMAL DRIVER
# Low PERCLOS, short eye closures, no yawns
# Expected SG-5 state: NORMAL
# ------------------------------------------------------------
normal_case = {
    "frame_id": 100,
    "timestamp_ms": 5000,
    "window_ms": 10000,
    "valid": True,
    "window_valid_fraction": 0.95,
    "perclos": 0.10,
    "blink_rate_per_min": 15.0,
    "mean_blink_duration_ms": 180.0,
    "longest_closure_ms": 300.0,
    "current_closure_ms": 0.0,
    "yawn_count": 0
}

# ------------------------------------------------------------
# CASE 2: WARNING-LEVEL DRIVER
# Moderately increased PERCLOS and eye-closure duration
# Expected SG-5 state: WARNING
# ------------------------------------------------------------
warning_case = {
    "frame_id": 101,
    "timestamp_ms": 6000,
    "window_ms": 10000,
    "valid": True,
    "window_valid_fraction": 0.92,
    "perclos": 0.24,
    "blink_rate_per_min": 18.0,
    "mean_blink_duration_ms": 240.0,
    "longest_closure_ms": 900.0,
    "current_closure_ms": 0.0,
    "yawn_count": 1
}

# ------------------------------------------------------------
# CASE 3: CLEARLY DROWSY DRIVER
# High PERCLOS, long closure and repeated yawns
# Expected SG-5 state: DROWSY
# ------------------------------------------------------------
drowsy_case = {
    "frame_id": 102,
    "timestamp_ms": 7000,
    "window_ms": 10000,
    "valid": True,
    "window_valid_fraction": 0.90,
    "perclos": 0.40,
    "blink_rate_per_min": 12.0,
    "mean_blink_duration_ms": 400.0,
    "longest_closure_ms": 1800.0,
    "current_closure_ms": 1500.0,
    "yawn_count": 3
}

# ------------------------------------------------------------
# CASE 4: PROLONGED EYE CLOSURE
# PERCLOS itself is not very high, but the current and longest
# eye closure are dangerously long
# Expected SG-5 state: DROWSY
# ------------------------------------------------------------
long_closure_case = {
    "frame_id": 103,
    "timestamp_ms": 8000,
    "window_ms": 10000,
    "valid": True,
    "window_valid_fraction": 0.94,
    "perclos": 0.18,
    "blink_rate_per_min": 14.0,
    "mean_blink_duration_ms": 200.0,
    "longest_closure_ms": 2200.0,
    "current_closure_ms": 2200.0,
    "yawn_count": 0
}

# ------------------------------------------------------------
# CASE 5: MULTIPLE MODERATE DROWSINESS CUES
# No single cue is extremely high, but several moderate cues
# occur together
# Expected SG-5 state: WARNING or DROWSY depending on method
# ------------------------------------------------------------
multiple_moderate_cues = {
    "frame_id": 104,
    "timestamp_ms": 9000,
    "window_ms": 10000,
    "valid": True,
    "window_valid_fraction": 0.91,
    "perclos": 0.28,
    "blink_rate_per_min": 19.0,
    "mean_blink_duration_ms": 300.0,
    "longest_closure_ms": 1000.0,
    "current_closure_ms": 0.0,
    "yawn_count": 2
}

# ------------------------------------------------------------
# CASE 6: INVALID TEMPORAL DATA
# SG-4 reports that there is not enough reliable data
# Measurement values are therefore None
# Expected SG-5 result: decision_valid = False
# ------------------------------------------------------------
invalid_case = {
    "frame_id": 105,
    "timestamp_ms": 10000,
    "window_ms": 10000,
    "valid": False,
    "window_valid_fraction": 0.30,
    "perclos": None,
    "blink_rate_per_min": None,
    "mean_blink_duration_ms": None,
    "longest_closure_ms": None,
    "current_closure_ms": None,
    "yawn_count": None
}

Overwriting mock_inputs.py


In [12]:
# ============================================================
# CELL 3: RULE-BASED SG-5 BASELINE
# ============================================================
# Strong cues produce DROWSY.
# Moderate cues produce WARNING.
# ============================================================

%%writefile decision_rules.py

def make_decision(data):

    # --------------------------------------------------------
    # STEP 1: VALIDATE SG-4 INPUT
    # --------------------------------------------------------
    if not data["valid"]:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    if data["window_valid_fraction"] < 0.70:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    # --------------------------------------------------------
    # STEP 2: EXTRACT FEATURES
    # --------------------------------------------------------
    perclos = data["perclos"]
    longest = data["longest_closure_ms"]
    current = data["current_closure_ms"]
    yawns = data["yawn_count"]

    # --------------------------------------------------------
    # STEP 3: STRONG DROWSINESS CONDITIONS
    # --------------------------------------------------------
    if (
        perclos >= 0.35
        or longest >= 1500
        or current >= 2000
        or yawns >= 3
    ):
        state = "DROWSY"
        score = 0.80

    else:
        # ----------------------------------------------------
        # STEP 4: MODERATE WARNING CONDITIONS
        # ----------------------------------------------------
        warning_count = 0

        if perclos >= 0.20:
            warning_count += 1

        if longest >= 800:
            warning_count += 1

        if yawns >= 2:
            warning_count += 1

        if warning_count >= 1:
            state = "WARNING"
            score = 0.50

        else:
            state = "NORMAL"
            score = 0.20

    # --------------------------------------------------------
    # STEP 5: RETURN SG-5 OUTPUT
    # --------------------------------------------------------
    return {
        "frame_id": data["frame_id"],
        "timestamp_ms": data["timestamp_ms"],
        "decision_valid": True,
        "state": state,
        "drowsiness_score": score,
        "alarm_flag": state == "DROWSY",
        "alarm_type": "DROWSINESS" if state == "DROWSY" else None
    }

Overwriting decision_rules.py


In [13]:
# ============================================================
# CELL 4: WEIGHTED-FUSION SG-5 METHOD
# ============================================================
# score < 0.30        -> NORMAL
# 0.30 to < 0.60     -> WARNING
# score >= 0.60       -> DROWSY
# ============================================================

%%writefile weighted_fusion.py

def weighted_decision(data):

    # --------------------------------------------------------
    # STEP 1: VALIDATE SG-4 INPUT
    # --------------------------------------------------------
    if not data["valid"]:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    if data["window_valid_fraction"] < 0.70:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    # --------------------------------------------------------
    # STEP 2: EXTRACT FEATURES
    # --------------------------------------------------------
    perclos = data["perclos"]
    longest = data["longest_closure_ms"]
    current = data["current_closure_ms"]
    yawns = data["yawn_count"]

    # --------------------------------------------------------
    # STEP 3: NORMALIZE FEATURES TO 0-1
    # --------------------------------------------------------
    perclos_score = min(max(perclos, 0.0), 1.0)

    closure_score = min(
        max(longest / 2000.0, 0.0),
        1.0
    )

    yawn_score = min(
        max(yawns / 3.0, 0.0),
        1.0
    )

    # --------------------------------------------------------
    # STEP 4: SAFETY OVERRIDE
    # --------------------------------------------------------
    if current >= 2000:
        score = 1.0
        state = "DROWSY"

    else:
        # ----------------------------------------------------
        # STEP 5: COMPUTE WEIGHTED SCORE
        # ----------------------------------------------------
        score = (
            0.50 * perclos_score
            + 0.30 * closure_score
            + 0.20 * yawn_score
        )

        score = min(max(score, 0.0), 1.0)

        # ----------------------------------------------------
        # STEP 6: CONVERT SCORE TO STATE
        # ----------------------------------------------------
        if score >= 0.60:
            state = "DROWSY"

        elif score >= 0.30:
            state = "WARNING"

        else:
            state = "NORMAL"

    # --------------------------------------------------------
    # STEP 7: RETURN SG-5 OUTPUT
    # --------------------------------------------------------
    return {
        "frame_id": data["frame_id"],
        "timestamp_ms": data["timestamp_ms"],
        "decision_valid": True,
        "state": state,
        "drowsiness_score": round(score, 3),
        "alarm_flag": state == "DROWSY",
        "alarm_type": "DROWSINESS" if state == "DROWSY" else None
    }

Overwriting weighted_fusion.py


In [10]:
# ============================================================
# CELL 5: COMPARE RULE-BASED AND WEIGHTED SG-5 METHODS
# ============================================================
# This script runs BOTH SG-5 decision methods on the SAME
# mock SG-4 inputs.
#
# Purpose:
# - Check that both methods are working
# - Compare their output states and scores
# - Start building evidence for Week 8 evaluation
# ============================================================

%%writefile compare_methods.py

from mock_inputs import (
    normal_case,
    warning_case,
    drowsy_case,
    long_closure_case,
    multiple_moderate_cues,
    invalid_case
)

from decision_rules import make_decision
from weighted_fusion import weighted_decision


# ------------------------------------------------------------
# TEST CASES
# Each case represents a different SG-4 temporal scenario.
# ------------------------------------------------------------
test_cases = {
    "Normal": normal_case,
    "Warning": warning_case,
    "Drowsy": drowsy_case,
    "Long Closure": long_closure_case,
    "Multiple Moderate Cues": multiple_moderate_cues,
    "Invalid": invalid_case
}


# ------------------------------------------------------------
# RUN BOTH METHODS ON THE SAME INPUT
# ------------------------------------------------------------
for name, data in test_cases.items():

    rule_result = make_decision(data)
    weighted_result = weighted_decision(data)

    print("=" * 60)
    print("Test Case:", name)

    print("\nRule-Based Method")
    print("State:", rule_result["state"])
    print("Score:", rule_result["drowsiness_score"])
    print("Alarm:", rule_result["alarm_flag"])

    print("\nWeighted-Fusion Method")
    print("State:", weighted_result["state"])
    print("Score:", weighted_result["drowsiness_score"])
    print("Alarm:", weighted_result["alarm_flag"])

    print()

Overwriting compare_methods.py


In [14]:
# ============================================================
# CELL 6: RUN THE SG-5 METHOD COMPARISON
# ============================================================

!python compare_methods.py

Test Case: Normal

Rule-Based Method
State: NORMAL
Score: 0.2
Alarm: False

Weighted-Fusion Method
State: NORMAL
Score: 0.095
Alarm: False

Test Case: Warning

Rule-Based Method
State: WARNING
Score: 0.5
Alarm: False

Weighted-Fusion Method
State: WARNING
Score: 0.322
Alarm: False

Test Case: Drowsy

Rule-Based Method
State: DROWSY
Score: 0.8
Alarm: True

Weighted-Fusion Method
State: DROWSY
Score: 0.67
Alarm: True

Test Case: Long Closure

Rule-Based Method
State: DROWSY
Score: 0.8
Alarm: True

Weighted-Fusion Method
State: DROWSY
Score: 1.0
Alarm: True

Test Case: Multiple Moderate Cues

Rule-Based Method
State: WARNING
Score: 0.5
Alarm: False

Weighted-Fusion Method
State: WARNING
Score: 0.423
Alarm: False

Test Case: Invalid

Rule-Based Method
State: None
Score: None
Alarm: False

Weighted-Fusion Method
State: None
Score: None
Alarm: False



In [15]:
# ============================================================
# CELL 7: QUANTITATIVE EVALUATION OF BOTH SG-5 METHODS
# ============================================================
# This cell:
# - defines the expected output for each mock test case
# - runs both decision methods
# - compares predictions with expected states
# - calculates simple classification accuracy
#
# This gives us preliminary quantitative evidence for Week 8.
# ============================================================

from mock_inputs import (
    normal_case,
    warning_case,
    drowsy_case,
    long_closure_case,
    multiple_moderate_cues,
    invalid_case
)

from decision_rules import make_decision
from weighted_fusion import weighted_decision


# ------------------------------------------------------------
# TEST CASES WITH EXPECTED OUTPUT STATES
# ------------------------------------------------------------
test_cases = {
    "Normal": (normal_case, "NORMAL"),
    "Warning": (warning_case, "WARNING"),
    "Drowsy": (drowsy_case, "DROWSY"),
    "Long Closure": (long_closure_case, "DROWSY"),
    "Multiple Moderate Cues": (multiple_moderate_cues, "WARNING"),
    "Invalid": (invalid_case, None)
}


rule_correct = 0
weighted_correct = 0
total_cases = len(test_cases)


# ------------------------------------------------------------
# RUN BOTH METHODS AND COMPARE WITH EXPECTED STATES
# ------------------------------------------------------------
for name, (data, expected_state) in test_cases.items():

    rule_result = make_decision(data)
    weighted_result = weighted_decision(data)

    rule_state = rule_result["state"]
    weighted_state = weighted_result["state"]

    rule_match = rule_state == expected_state
    weighted_match = weighted_state == expected_state

    if rule_match:
        rule_correct += 1

    if weighted_match:
        weighted_correct += 1

    print("=" * 65)
    print("Test Case:", name)
    print("Expected:", expected_state)

    print(
        "Rule-Based:",
        rule_state,
        "| Correct:",
        rule_match
    )

    print(
        "Weighted:",
        weighted_state,
        "| Correct:",
        weighted_match
    )


# ------------------------------------------------------------
# CALCULATE SIMPLE ACCURACY
# ------------------------------------------------------------
rule_accuracy = (rule_correct / total_cases) * 100
weighted_accuracy = (weighted_correct / total_cases) * 100


print("\n" + "=" * 65)
print("SG-5 PRELIMINARY RESULTS")
print("=" * 65)

print(
    f"Rule-Based Accuracy: {rule_correct}/{total_cases} "
    f"= {rule_accuracy:.1f}%"
)

print(
    f"Weighted-Fusion Accuracy: {weighted_correct}/{total_cases} "
    f"= {weighted_accuracy:.1f}%"
)

Test Case: Normal
Expected: NORMAL
Rule-Based: NORMAL | Correct: True
Weighted: NORMAL | Correct: True
Test Case: Warning
Expected: WARNING
Rule-Based: WARNING | Correct: True
Weighted: WARNING | Correct: True
Test Case: Drowsy
Expected: DROWSY
Rule-Based: DROWSY | Correct: True
Weighted: DROWSY | Correct: True
Test Case: Long Closure
Expected: DROWSY
Rule-Based: DROWSY | Correct: True
Weighted: DROWSY | Correct: True
Test Case: Multiple Moderate Cues
Expected: WARNING
Rule-Based: WARNING | Correct: True
Weighted: WARNING | Correct: True
Test Case: Invalid
Expected: None
Rule-Based: None | Correct: True
Weighted: None | Correct: True

SG-5 PRELIMINARY RESULTS
Rule-Based Accuracy: 6/6 = 100.0%
Weighted-Fusion Accuracy: 6/6 = 100.0%


In [17]:
# ============================================================
# CELL 8: ADD HARDER / BORDERLINE SG-5 TEST CASES
# ============================================================
# These cases are closer to the decision thresholds.
# They help us identify where the two methods disagree
# or where the current thresholds may fail.
# ============================================================

borderline_cases = {
    # Just below warning PERCLOS threshold
    "PERCLOS Below Warning": ({
        "frame_id": 200,
        "timestamp_ms": 11000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.95,
        "perclos": 0.19,
        "blink_rate_per_min": 16.0,
        "mean_blink_duration_ms": 190.0,
        "longest_closure_ms": 500.0,
        "current_closure_ms": 0.0,
        "yawn_count": 0
    }, "NORMAL"),

    # Just above warning PERCLOS threshold
    "PERCLOS Above Warning": ({
        "frame_id": 201,
        "timestamp_ms": 12000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.95,
        "perclos": 0.21,
        "blink_rate_per_min": 17.0,
        "mean_blink_duration_ms": 200.0,
        "longest_closure_ms": 500.0,
        "current_closure_ms": 0.0,
        "yawn_count": 0
    }, "WARNING"),

    # Just below strong closure threshold
    "Closure Below Drowsy": ({
        "frame_id": 202,
        "timestamp_ms": 13000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.94,
        "perclos": 0.15,
        "blink_rate_per_min": 14.0,
        "mean_blink_duration_ms": 220.0,
        "longest_closure_ms": 1450.0,
        "current_closure_ms": 0.0,
        "yawn_count": 0
    }, "WARNING"),

    # Just above strong closure threshold
    "Closure Above Drowsy": ({
        "frame_id": 203,
        "timestamp_ms": 14000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.94,
        "perclos": 0.15,
        "blink_rate_per_min": 14.0,
        "mean_blink_duration_ms": 220.0,
        "longest_closure_ms": 1550.0,
        "current_closure_ms": 0.0,
        "yawn_count": 0
    }, "DROWSY"),

    # Repeated yawns but normal eye behaviour
    "Yawns Only": ({
        "frame_id": 204,
        "timestamp_ms": 15000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.93,
        "perclos": 0.10,
        "blink_rate_per_min": 15.0,
        "mean_blink_duration_ms": 180.0,
        "longest_closure_ms": 400.0,
        "current_closure_ms": 0.0,
        "yawn_count": 2
    }, "WARNING"),

    # Poor temporal coverage
    "Low Valid Fraction": ({
        "frame_id": 205,
        "timestamp_ms": 16000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.60,
        "perclos": 0.40,
        "blink_rate_per_min": 10.0,
        "mean_blink_duration_ms": 350.0,
        "longest_closure_ms": 1700.0,
        "current_closure_ms": 1000.0,
        "yawn_count": 2
    }, None),

    # Multiple moderate cues close to the boundary
    "Moderate Combination": ({
        "frame_id": 206,
        "timestamp_ms": 17000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.92,
        "perclos": 0.22,
        "blink_rate_per_min": 18.0,
        "mean_blink_duration_ms": 250.0,
        "longest_closure_ms": 850.0,
        "current_closure_ms": 0.0,
        "yawn_count": 1
    }, "WARNING"),

    # High PERCLOS without yawns
    "High PERCLOS Only": ({
        "frame_id": 207,
        "timestamp_ms": 18000,
        "window_ms": 10000,
        "valid": True,
        "window_valid_fraction": 0.96,
        "perclos": 0.38,
        "blink_rate_per_min": 13.0,
        "mean_blink_duration_ms": 300.0,
        "longest_closure_ms": 700.0,
        "current_closure_ms": 0.0,
        "yawn_count": 0
    }, "DROWSY")
}

In [18]:
# ============================================================
# CELL 9: EVALUATE BOTH METHODS ON BORDERLINE TEST CASES
# ============================================================
# This cell:
# - runs the rule-based and weighted methods
# - compares predictions with expected states
# - calculates accuracy on the harder test set
# - shows where the methods disagree or fail
# ============================================================

from decision_rules import make_decision
from weighted_fusion import weighted_decision


rule_correct = 0
weighted_correct = 0
total_cases = len(borderline_cases)


for name, (data, expected_state) in borderline_cases.items():

    rule_result = make_decision(data)
    weighted_result = weighted_decision(data)

    rule_state = rule_result["state"]
    weighted_state = weighted_result["state"]

    rule_match = rule_state == expected_state
    weighted_match = weighted_state == expected_state

    if rule_match:
        rule_correct += 1

    if weighted_match:
        weighted_correct += 1

    print("=" * 70)
    print("Test Case:", name)
    print("Expected:", expected_state)

    print(
        "Rule-Based:",
        rule_state,
        "| Score:",
        rule_result["drowsiness_score"],
        "| Correct:",
        rule_match
    )

    print(
        "Weighted:",
        weighted_state,
        "| Score:",
        weighted_result["drowsiness_score"],
        "| Correct:",
        weighted_match
    )


# ------------------------------------------------------------
# CALCULATE ACCURACY
# ------------------------------------------------------------
rule_accuracy = (rule_correct / total_cases) * 100
weighted_accuracy = (weighted_correct / total_cases) * 100


print("\n" + "=" * 70)
print("BORDERLINE TEST RESULTS")
print("=" * 70)

print(
    f"Rule-Based Accuracy: {rule_correct}/{total_cases} "
    f"= {rule_accuracy:.1f}%"
)

print(
    f"Weighted-Fusion Accuracy: {weighted_correct}/{total_cases} "
    f"= {weighted_accuracy:.1f}%"
)

Test Case: PERCLOS Below Warning
Expected: NORMAL
Rule-Based: NORMAL | Score: 0.2 | Correct: True
Weighted: NORMAL | Score: 0.17 | Correct: True
Test Case: PERCLOS Above Warning
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.18 | Correct: False
Test Case: Closure Below Drowsy
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.292 | Correct: False
Test Case: Closure Above Drowsy
Expected: DROWSY
Rule-Based: DROWSY | Score: 0.8 | Correct: True
Weighted: WARNING | Score: 0.307 | Correct: False
Test Case: Yawns Only
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.243 | Correct: False
Test Case: Low Valid Fraction
Expected: None
Rule-Based: None | Score: None | Correct: True
Weighted: None | Score: None | Correct: True
Test Case: Moderate Combination
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: WARNING | Score: 0.304 | Cor

In [19]:
# ============================================================
# CELL 10: HYBRID WEIGHTED-FUSION SG-5 METHOD
# ============================================================
# This method combines:
# 1. Direct thresholds for important individual cues
# 2. Weighted fusion for multiple moderate cues
#
# This prevents strong single cues from being diluted by
# averaging while still allowing multi-cue evidence to combine.
# ============================================================

%%writefile hybrid_fusion.py

def hybrid_decision(data):

    # --------------------------------------------------------
    # STEP 1: VALIDATE SG-4 INPUT
    # --------------------------------------------------------
    if not data["valid"]:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    if data["window_valid_fraction"] < 0.70:
        return {
            "frame_id": data["frame_id"],
            "timestamp_ms": data["timestamp_ms"],
            "decision_valid": False,
            "state": None,
            "drowsiness_score": None,
            "alarm_flag": False,
            "alarm_type": None
        }

    # --------------------------------------------------------
    # STEP 2: EXTRACT FEATURES
    # --------------------------------------------------------
    perclos = data["perclos"]
    longest = data["longest_closure_ms"]
    current = data["current_closure_ms"]
    yawns = data["yawn_count"]

    # --------------------------------------------------------
    # STEP 3: STRONG SAFETY CONDITIONS
    # --------------------------------------------------------
    # Strong individual cues can directly produce DROWSY.
    if (
        perclos >= 0.35
        or longest >= 1500
        or current >= 2000
        or yawns >= 3
    ):
        state = "DROWSY"
        score = 0.80

    else:

        # ----------------------------------------------------
        # STEP 4: NORMALIZE FEATURES TO 0-1
        # ----------------------------------------------------
        perclos_score = min(max(perclos, 0.0), 1.0)

        closure_score = min(
            max(longest / 2000.0, 0.0),
            1.0
        )

        yawn_score = min(
            max(yawns / 3.0, 0.0),
            1.0
        )

        # ----------------------------------------------------
        # STEP 5: CALCULATE WEIGHTED SCORE
        # ----------------------------------------------------
        score = (
            0.50 * perclos_score
            + 0.30 * closure_score
            + 0.20 * yawn_score
        )

        score = min(max(score, 0.0), 1.0)

        # ----------------------------------------------------
        # STEP 6: CHECK MODERATE INDIVIDUAL CUES
        # ----------------------------------------------------
        moderate_cue = (
            perclos >= 0.20
            or longest >= 800
            or yawns >= 2
        )

        # ----------------------------------------------------
        # STEP 7: ASSIGN STATE
        # ----------------------------------------------------
        # Moderate individual evidence or sufficient combined
        # weighted evidence produces WARNING.
        if moderate_cue or score >= 0.30:
            state = "WARNING"

        else:
            state = "NORMAL"

    # --------------------------------------------------------
    # STEP 8: RETURN SG-5 OUTPUT
    # --------------------------------------------------------
    return {
        "frame_id": data["frame_id"],
        "timestamp_ms": data["timestamp_ms"],
        "decision_valid": True,
        "state": state,
        "drowsiness_score": round(score, 3),
        "alarm_flag": state == "DROWSY",
        "alarm_type": "DROWSINESS" if state == "DROWSY" else None
    }

Writing hybrid_fusion.py


In [20]:
# ============================================================
# CELL 11: COMPARE ALL THREE SG-5 METHODS
# ============================================================
# This cell compares:
# 1. Rule-based method
# 2. Weighted-fusion method
# 3. Hybrid-fusion method
#
# It runs all three on the harder borderline test cases
# and compares their predictions against expected states.
# ============================================================

from decision_rules import make_decision
from weighted_fusion import weighted_decision
from hybrid_fusion import hybrid_decision


rule_correct = 0
weighted_correct = 0
hybrid_correct = 0

total_cases = len(borderline_cases)


for name, (data, expected_state) in borderline_cases.items():

    rule_result = make_decision(data)
    weighted_result = weighted_decision(data)
    hybrid_result = hybrid_decision(data)

    rule_state = rule_result["state"]
    weighted_state = weighted_result["state"]
    hybrid_state = hybrid_result["state"]

    rule_match = rule_state == expected_state
    weighted_match = weighted_state == expected_state
    hybrid_match = hybrid_state == expected_state

    if rule_match:
        rule_correct += 1

    if weighted_match:
        weighted_correct += 1

    if hybrid_match:
        hybrid_correct += 1

    print("=" * 75)
    print("Test Case:", name)
    print("Expected:", expected_state)

    print(
        "Rule-Based:",
        rule_state,
        "| Score:",
        rule_result["drowsiness_score"],
        "| Correct:",
        rule_match
    )

    print(
        "Weighted:",
        weighted_state,
        "| Score:",
        weighted_result["drowsiness_score"],
        "| Correct:",
        weighted_match
    )

    print(
        "Hybrid:",
        hybrid_state,
        "| Score:",
        hybrid_result["drowsiness_score"],
        "| Correct:",
        hybrid_match
    )


# ------------------------------------------------------------
# CALCULATE ACCURACY
# ------------------------------------------------------------
rule_accuracy = (rule_correct / total_cases) * 100
weighted_accuracy = (weighted_correct / total_cases) * 100
hybrid_accuracy = (hybrid_correct / total_cases) * 100


print("\n" + "=" * 75)
print("SG-5 METHOD COMPARISON RESULTS")
print("=" * 75)

print(
    f"Rule-Based Accuracy: {rule_correct}/{total_cases} "
    f"= {rule_accuracy:.1f}%"
)

print(
    f"Weighted-Fusion Accuracy: {weighted_correct}/{total_cases} "
    f"= {weighted_accuracy:.1f}%"
)

print(
    f"Hybrid-Fusion Accuracy: {hybrid_correct}/{total_cases} "
    f"= {hybrid_accuracy:.1f}%"
)

Test Case: PERCLOS Below Warning
Expected: NORMAL
Rule-Based: NORMAL | Score: 0.2 | Correct: True
Weighted: NORMAL | Score: 0.17 | Correct: True
Hybrid: NORMAL | Score: 0.17 | Correct: True
Test Case: PERCLOS Above Warning
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.18 | Correct: False
Hybrid: WARNING | Score: 0.18 | Correct: True
Test Case: Closure Below Drowsy
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.292 | Correct: False
Hybrid: WARNING | Score: 0.292 | Correct: True
Test Case: Closure Above Drowsy
Expected: DROWSY
Rule-Based: DROWSY | Score: 0.8 | Correct: True
Weighted: WARNING | Score: 0.307 | Correct: False
Hybrid: DROWSY | Score: 0.8 | Correct: True
Test Case: Yawns Only
Expected: WARNING
Rule-Based: WARNING | Score: 0.5 | Correct: True
Weighted: NORMAL | Score: 0.243 | Correct: False
Hybrid: WARNING | Score: 0.243 | Correct: True
Test Case: Low Valid Fraction
Expected: None


In [21]:
# ============================================================
# CELL 12: CREATE A CLEAN SG-5 COMPARISON TABLE
# ============================================================
# This cell creates a readable table showing:
# - test case
# - expected state
# - rule-based result
# - weighted-fusion result
# - hybrid-fusion result
#
# This table can be shown during the Week 8 evaluation.
# ============================================================

import pandas as pd

from decision_rules import make_decision
from weighted_fusion import weighted_decision
from hybrid_fusion import hybrid_decision


results = []

for name, (data, expected_state) in borderline_cases.items():

    rule_result = make_decision(data)
    weighted_result = weighted_decision(data)
    hybrid_result = hybrid_decision(data)

    results.append({
        "Test Case": name,
        "Expected": expected_state,
        "Rule-Based": rule_result["state"],
        "Weighted Fusion": weighted_result["state"],
        "Hybrid Fusion": hybrid_result["state"]
    })


# Create dataframe
df_results = pd.DataFrame(results)

# Display table
display(df_results)


# ------------------------------------------------------------
# SAVE TABLE AS CSV
# ------------------------------------------------------------
df_results.to_csv("sg5_method_comparison.csv", index=False)

print("\nSaved as: sg5_method_comparison.csv")

,Test Case,Expected,Rule-Based,Weighted Fusion,Hybrid Fusion
0,PERCLOS Below Warning,NORMAL,NORMAL,NORMAL,NORMAL
1,PERCLOS Above Warning,WARNING,WARNING,NORMAL,WARNING
2,Closure Below Drowsy,WARNING,WARNING,NORMAL,WARNING
3,Closure Above Drowsy,DROWSY,DROWSY,WARNING,DROWSY
4,Yawns Only,WARNING,WARNING,NORMAL,WARNING
5,Low Valid Fraction,None,None,None,None
6,Moderate Combination,WARNING,WARNING,WARNING,WARNING
7,High PERCLOS Only,DROWSY,DROWSY,NORMAL,DROWSY



Saved as: sg5_method_comparison.csv
